In [21]:
import pandas as pd
import numpy as np
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

In [22]:
from pathlib import Path
root = Path.cwd()
if not (root / "data" / "distributors.csv").exists():
    root = root.parent

data_dir = root / "data"

In [23]:
distributor = pd.read_csv(data_dir/"distributors.csv")
orders = pd.read_csv(data_dir/"orders.csv")
payments = pd.read_csv(data_dir/"payments.csv")

In [24]:
distributor.head()

,distributor_id,distributor_name,region,city,channel,credit_limit_inr,credit_terms_days,onboarded_date
0,DIST-001,Om Brothers,North,Delhi,institution,300000,21,2022-04-03
1,DIST-002,Modern Traders,North,Chandigarh,institution,750000,15,2024-04-24
2,DIST-003,Jai Brothers,North,Delhi,modern_trade,200000,15,2024-02-24
3,DIST-004,Om Brothers,West,Mumbai,general_trade,750000,30,2021-01-14
4,DIST-005,New Traders,West,Pune,general_trade,200000,21,2025-04-13


In [25]:
orders.head()

,order_id,distributor_id,order_date,order_value_inr,order_status
0,ORD-00001,DIST-001,2026-01-11,25000.0,delivered
1,ORD-00002,DIST-001,2026-04-20,150000.0,cancelled
2,ORD-00003,DIST-001,2024-07-12,50000.0,delivered
3,ORD-00004,DIST-001,2025-09-25,100000.0,delivered
4,ORD-00005,DIST-001,2025-03-12,150000.0,delivered


In [26]:
payments.head()

,payment_id,distributor_id,invoice_id,invoice_date,invoice_amount_inr,due_date,payment_date,amount_paid_inr,payment_status
0,PAY-000717,DIST-063,INV-000717,2025-09-29,257488,2025-10-29,2025-11-18,257488,paid
1,PAY-000757,DIST-067,INV-000757,2024-08-01,333376,2024-08-22,2024-10-01,333376,paid
2,PAY-000523,DIST-046,INV-000523,2025-07-02,47886,2025-07-17,2025-07-16,47886,paid
3,PAY-001173,DIST-105,INV-001173,2025-02-17,32931,2025-03-04,NaN,0,overdue
4,PAY-000678,DIST-060,INV-000678,2026-03-22,104443,2026-05-06,2026-06-03,104443,paid


# Phase 1 - EDA

Goal: find every data problem and behavioural pattern *before* writing validation or feature code.
Each section ends with what we learned. Nothing here modifies the CSVs.

In [28]:
payments[payments['payment_date'].isna()].head()

,payment_id,distributor_id,invoice_id,invoice_date,invoice_amount_inr,due_date,payment_date,amount_paid_inr,payment_status
3,PAY-001173,DIST-105,INV-001173,2025-02-17,32931,2025-03-04,NaN,0,overdue
14,PAY-001045,DIST-094,INV-001045,2024-12-17,281489,2025-01-16,NaN,0,overdue
52,PAY-001105,DIST-099,INV-001105,2024-08-21,26821,2024-10-05,NaN,0,overdue
55,PAY-001035,DIST-093,INV-001035,2025-04-17,236985,2025-06-01,NaN,0,overdue
57,PAY-001027,DIST-092,INV-001027,2025-05-02,178981,2025-06-01,NaN,0,overdue


In [29]:
distributor['onboarded_date'] = pd.to_datetime(distributor['onboarded_date'], format="%Y-%m-%d", errors="coerce")
orders['order_date'] = pd.to_datetime(orders['order_date'], format="%Y-%m-%d", errors="coerce")
for col in ["invoice_date", "due_date", "payment_date"]:
    payments[col] = pd.to_datetime(payments[col], format = "%Y-%m-%d", errors="coerce")

In [30]:
distributor['onboarded_date'].isna().sum()
orders['order_date'].isna().sum()
payments[['invoice_date', 'due_date']].isna().sum().sum()
payments['payment_date'].isna().sum()

np.int64(120)

In [32]:
payments[payments['payment_date'].isna()].head()

,payment_id,distributor_id,invoice_id,invoice_date,invoice_amount_inr,due_date,payment_date,amount_paid_inr,payment_status
3,PAY-001173,DIST-105,INV-001173,2025-02-17,32931,2025-03-04,NaT,0,overdue
14,PAY-001045,DIST-094,INV-001045,2024-12-17,281489,2025-01-16,NaT,0,overdue
52,PAY-001105,DIST-099,INV-001105,2024-08-21,26821,2024-10-05,NaT,0,overdue
55,PAY-001035,DIST-093,INV-001035,2025-04-17,236985,2025-06-01,NaT,0,overdue
57,PAY-001027,DIST-092,INV-001027,2025-05-02,178981,2025-06-01,NaT,0,overdue


In [33]:
# 2. Distributors: value domains and suspicious rows
print(distributor.credit_limit_inr.value_counts().sort_index().to_dict())
print(distributor.credit_terms_days.value_counts().sort_index().to_dict())
print("\nzero / non-positive credit limit (breaks utilisation = exposure / limit):")
display(distributor[distributor.credit_limit_inr <= 0])

# What does that distributor actually do?
zid = distributor.loc[distributor.credit_limit_inr <= 0, "distributor_id"].iloc[0]
print(zid, "orders:", (orders.distributor_id == zid).sum(),
      "| payments:", (payments.distributor_id == zid).sum(),
      "| statuses:", payments[payments.distributor_id == zid].payment_status.value_counts().to_dict())

print("\nchannel/region mix:")
print(distributor.channel.value_counts().to_dict(), distributor.region.value_counts().to_dict())
print("distributor_name is NOT unique (names repeat across cities):",
      distributor.distributor_name.nunique(), "unique of", len(distributor))

{0: 1, 100000: 22, 200000: 19, 300000: 18, 500000: 20, 750000: 22, 1000000: 18}
{15: 29, 21: 32, 30: 32, 45: 27}

zero / non-positive credit limit (breaks utilisation = exposure / limit):


,distributor_id,distributor_name,region,city,channel,credit_limit_inr,credit_terms_days,onboarded_date
14,DIST-015,Zero Limit Traders,East,Patna,institution,0,45,2023-01-12


DIST-015 orders: 22 | payments: 15 | statuses: {'paid': 15}

channel/region mix:
{'institution': 43, 'modern_trade': 41, 'general_trade': 36} {'East': 36, 'North': 29, 'West': 28, 'South': 27}
distributor_name is NOT unique (names repeat across cities): 53 unique of 120


In [38]:
distributor.loc[distributor.credit_limit_inr <= 0].head()

,distributor_id,distributor_name,region,city,channel,credit_limit_inr,credit_terms_days,onboarded_date
14,DIST-015,Zero Limit Traders,East,Patna,institution,0,45,2023-01-12


In [36]:
orders[orders.distributor_id == distributor.loc[distributor.credit_limit_inr <= 0, "distributor_id"].iloc[0]].head()

,order_id,distributor_id,order_date,order_value_inr,order_status
192,ORD-00193,DIST-015,2025-10-12,75000.0,delivered
193,ORD-00194,DIST-015,2024-07-17,150000.0,delivered
194,ORD-00195,DIST-015,2024-07-27,75000.0,delivered
195,ORD-00196,DIST-015,2025-01-03,300000.0,delivered
196,ORD-00197,DIST-015,2025-02-17,25000.0,delivered


In [37]:
payments[payments.distributor_id == distributor.loc[distributor.credit_limit_inr <= 0, "distributor_id"].iloc[0]].head()

,payment_id,distributor_id,invoice_id,invoice_date,invoice_amount_inr,due_date,payment_date,amount_paid_inr,payment_status
21,PAY-000174,DIST-015,INV-000174,2025-10-28,28359,2025-12-12,2025-12-10,28359,paid
48,PAY-000167,DIST-015,INV-000167,2025-02-16,27519,2025-04-02,2025-03-30,27519,paid
76,PAY-000162,DIST-015,INV-000162,2024-08-27,16946,2024-10-11,2024-10-10,16946,paid
235,PAY-000164,DIST-015,INV-000164,2024-10-25,28535,2024-12-09,2024-12-08,28535,paid
251,PAY-000173,DIST-015,INV-000173,2025-09-20,25939,2025-11-04,2025-11-03,25939,paid


In [39]:
orders.head()

,order_id,distributor_id,order_date,order_value_inr,order_status
0,ORD-00001,DIST-001,2026-01-11,25000.0,delivered
1,ORD-00002,DIST-001,2026-04-20,150000.0,cancelled
2,ORD-00003,DIST-001,2024-07-12,50000.0,delivered
3,ORD-00004,DIST-001,2025-09-25,100000.0,delivered
4,ORD-00005,DIST-001,2025-03-12,150000.0,delivered


In [40]:
# 3. Orders: value distribution, status, and "order before onboarding"
print(orders.order_value_inr.describe().round(0).to_dict())
print("distinct order values:", sorted(orders.order_value_inr.dropna().unique()))
print(orders.order_status.value_counts().to_dict())

m = orders.merge(distributor[["distributor_id", "onboarded_date"]], on="distributor_id")
before = m[m.order_date < m.onboarded_date]
print(f"\norders dated BEFORE onboarding: {len(before)} of {len(m)} "
      f"({before.distributor_id.nunique()} distributors)")
print("days before onboarding:", (before.onboarded_date - before.order_date).dt.days.describe().round(0).to_dict())

print("\npending orders by quarter (do old ones make sense as 'pending'?):")
print(orders[orders.order_status == "pending"].order_date.dt.to_period("Q").value_counts().sort_index().to_dict())

{'count': 1815.0, 'mean': 131570.0, 'std': 88522.0, 'min': 25000.0, '25%': 50000.0, '50%': 100000.0, '75%': 200000.0, 'max': 300000.0}
distinct order values: [np.float64(25000.0), np.float64(50000.0), np.float64(75000.0), np.float64(100000.0), np.float64(150000.0), np.float64(200000.0), np.float64(300000.0)]
{'delivered': 1454, 'cancelled': 184, 'pending': 181}

orders dated BEFORE onboarding: 135 of 1819 (28 distributors)
days before onboarding: {'count': 135.0, 'mean': 127.0, 'std': 84.0, 'min': 4.0, '25%': 54.0, '50%': 110.0, '75%': 186.0, 'max': 344.0}

pending orders by quarter (do old ones make sense as 'pending'?):
{Period('2024Q2', 'Q-DEC'): 7, Period('2024Q3', 'Q-DEC'): 22, Period('2024Q4', 'Q-DEC'): 19, Period('2025Q1', 'Q-DEC'): 16, Period('2025Q2', 'Q-DEC'): 28, Period('2025Q3', 'Q-DEC'): 21, Period('2025Q4', 'Q-DEC'): 26, Period('2026Q1', 'Q-DEC'): 21, Period('2026Q2', 'Q-DEC'): 21}


In [41]:
m.head()

,order_id,distributor_id,order_date,order_value_inr,order_status,onboarded_date
0,ORD-00001,DIST-001,2026-01-11,25000.0,delivered,2022-04-03
1,ORD-00002,DIST-001,2026-04-20,150000.0,cancelled,2022-04-03
2,ORD-00003,DIST-001,2024-07-12,50000.0,delivered,2022-04-03
3,ORD-00004,DIST-001,2025-09-25,100000.0,delivered,2022-04-03
4,ORD-00005,DIST-001,2025-03-12,150000.0,delivered,2022-04-03


In [42]:
before.head()

,order_id,distributor_id,order_date,order_value_inr,order_status,onboarded_date
64,ORD-00065,DIST-005,2024-11-22,150000.0,delivered,2025-04-13
67,ORD-00068,DIST-005,2025-01-24,75000.0,delivered,2025-04-13
141,ORD-00142,DIST-010,2024-07-15,300000.0,delivered,2024-12-08
148,ORD-00149,DIST-010,2024-10-08,200000.0,delivered,2024-12-08
153,ORD-00154,DIST-010,2024-09-26,200000.0,cancelled,2024-12-08


In [43]:
distributor[distributor.distributor_id == 'DIST-005'].head()

,distributor_id,distributor_name,region,city,channel,credit_limit_inr,credit_terms_days,onboarded_date
4,DIST-005,New Traders,West,Pune,general_trade,200000,21,2025-04-13


In [44]:
payments.head()

,payment_id,distributor_id,invoice_id,invoice_date,invoice_amount_inr,due_date,payment_date,amount_paid_inr,payment_status
0,PAY-000717,DIST-063,INV-000717,2025-09-29,257488,2025-10-29,2025-11-18,257488,paid
1,PAY-000757,DIST-067,INV-000757,2024-08-01,333376,2024-08-22,2024-10-01,333376,paid
2,PAY-000523,DIST-046,INV-000523,2025-07-02,47886,2025-07-17,2025-07-16,47886,paid
3,PAY-001173,DIST-105,INV-001173,2025-02-17,32931,2025-03-04,NaT,0,overdue
4,PAY-000678,DIST-060,INV-000678,2026-03-22,104443,2026-05-06,2026-06-03,104443,paid


In [ ]:
# 4. Payments: date logic, amount logic, status vs amounts
pay = payments[payments.distributor_id.isin(known)].copy()          # orphans excluded
pay["terms_gap"] = (pay.due_date - pay.invoice_date).dt.days
pay["paid_ratio"] = pay.amount_paid_inr / pay.invoice_amount_inr
pay["days_late"] = (pay.payment_date - pay.due_date).dt.days        # NaN if unpaid

print("due <= invoice:", (pay.due_date <= pay.invoice_date).sum())
print("payment BEFORE invoice (impossible):")
display(pay[pay.payment_date < pay.invoice_date])
print("overpaid (paid > invoice):")
display(pay[pay.paid_ratio > 1])

print("\nstatus vs amounts - do statuses agree with the numbers?")
display(pay.groupby("payment_status").agg(
    n=("payment_id", "size"),
    paid_ratio_min=("paid_ratio", "min"), paid_ratio_max=("paid_ratio", "max"),
    missing_payment_date=("payment_date", lambda s: s.isna().sum()),
    zero_paid=("amount_paid_inr", lambda s: (s == 0).sum()),
))

# Does the invoice's due-date gap match the distributor's contractual terms?
chk = pay.merge(distributor[["distributor_id", "credit_terms_days", "onboarded_date"]], on="distributor_id")
mism = chk[chk.terms_gap != chk.credit_terms_days]
print(f"\nterms gap != contractual terms: {len(mism)} rows, distributors: {mism.distributor_id.unique().tolist()}")
print("invoice date range for those rows:", mism.invoice_date.min().date(), "->", mism.invoice_date.max().date())
print("invoices dated before onboarding:", (chk.invoice_date < chk.onboarded_date).sum())

In [ ]:
# 5. Time horizon: where does the data "end"? (decides our as-of date)
print("last order_date  :", orders.order_date.max().date())
print("last invoice_date:", pay.invoice_date.max().date())
print("last payment_date:", pay.payment_date.max().date(), "(payments dated after the last order/invoice?)")
print("payments dated after 2026-06-30:", (pay.payment_date > "2026-06-30").sum())
display(pay[pay.payment_date > "2026-06-30"][["payment_id", "invoice_date", "due_date", "payment_date"]])

print("\ninvoices per quarter:", pay.groupby(pay.invoice_date.dt.to_period("Q")).size().to_dict())
print("orders   per quarter:", orders.groupby(orders.order_date.dt.to_period("Q")).size().to_dict())
print("\noverdue invoices by due quarter:",
      pay[pay.payment_status == "overdue"].due_date.dt.to_period("Q").value_counts().sort_index().to_dict())
print("write-offs by invoice quarter:",
      pay[pay.payment_status == "written_off"].invoice_date.dt.to_period("Q").value_counts().sort_index().to_dict())

In [ ]:
# 6. Behaviour: how does payment behaviour vary across distributors?
pay["dist_no"] = pay.distributor_id.str[-3:].astype(int)
pay["id_block"] = pd.cut(pay.dist_no, [0, 30, 60, 90, 101, 120],
                         labels=["001-030", "031-060", "061-090", "091-101", "102-120"])
display(pay.groupby("id_block", observed=True).agg(
    distributors=("distributor_id", "nunique"), invoices=("payment_id", "size"),
    overdue=("payment_status", lambda s: (s == "overdue").mean()),
    partial=("payment_status", lambda s: (s == "partial").mean()),
    written_off=("payment_status", lambda s: (s == "written_off").mean()),
    mean_days_late=("days_late", "mean"),
).round(3))

per_dist = pay.groupby("distributor_id").agg(
    invoices=("payment_id", "size"),
    n_overdue=("payment_status", lambda s: (s == "overdue").sum()),
    n_partial=("payment_status", lambda s: (s == "partial").sum()),
    n_written_off=("payment_status", lambda s: (s == "written_off").sum()),
    avg_days_late=("days_late", "mean"),
)
print("distributors with >=1 overdue / partial / written_off:",
      (per_dist.n_overdue > 0).sum(), (per_dist.n_partial > 0).sum(), (per_dist.n_written_off > 0).sum())

# Exposure: unpaid balance on invoices that are open (overdue/partial), vs credit limit
pay["unpaid"] = pay.invoice_amount_inr - pay.amount_paid_inr.clip(upper=pay.invoice_amount_inr)
open_exp = pay[pay.payment_status.isin(["overdue", "partial"])].groupby("distributor_id").unpaid.sum()
util = (open_exp / distributor.set_index("distributor_id").credit_limit_inr.replace(0, np.nan)).dropna()
print("distributors with open exposure > credit limit:", (util > 1).sum(), "of", len(util))

# Do orders and invoices reconcile?
print("\ncorr(#orders, #invoices) per distributor:",
      orders.groupby("distributor_id").size().corr(pay.groupby("distributor_id").size()).round(3))
print("payments has no order_id -> invoices cannot be tied to specific orders")

## Issue register (what the validation rules in Phase 2 must handle)

| # | Issue | Where | Count |
|---|---|---|---|
| 1 | Orphan payments (`DIST-900..904`, ids `*-TRAP-*`) | payments | 5 |
| 2 | Null `order_value_inr` | orders | 4 |
| 3 | Zero credit limit (`DIST-015`) -> utilisation divides by zero | distributors | 1 |
| 4 | Orders dated before onboarding | orders | 135 (28 distributors) |
| 5 | Invoices dated before onboarding | payments | 85 |
| 6 | Payment date before invoice date | payments | 3 |
| 7 | Overpayment (paid > invoice, up to 1.5x) | payments | 2 |
| 8 | Due-date gap != contractual terms (`DIST-045`, 30 vs 21 days, Jan-May 2024) | payments | 15 |
| 9 | Payment dates after the data horizon (July 2026) | payments | 2 to 7 depending on cutoff |
| 10 | Distributors with no orders and no payments (`DIST-119`, `DIST-120`) | all | 2 |

Decisions still open (see notes in chat): as-of date, treatment of #4/#5, #8, #9, and what "overdue" means relative to the as-of date.